# Data Journalism Project: Analysis of Traffic Accidents in Serbia (2015-2026)

## Context and Research Objectives
In Serbia, an average of around 500 people lose their lives in traffic accidents every year. Per capita, this grim statistic places Serbia at the very top of European black lists regarding traffic safety. 

This data-driven journalism project aims to deeply analyze structural patterns, main causes, and demographic factors behind these accidents over an 11-year period (2015–2026). 

### Key Research Questions:
1. **Primary Causes:** What are the most frequent triggers and behavioral causes of fatal traffic accidents?
2. **Demographics & Substance Abuse:** What is the age distribution of the perpetrators versus the victims (average age of the deceased)? What percentage of fatal accidents involves driving under the influence of alcohol or other illicit substances?
3. **Temporal Patterns:** How do fatalities fluctuate across different months of the year, days of the week, or hours of the day? Are there specific seasonal peaks?
4. **Geographical Hotspots ("Black Spots"):** Which Police Directorates (Policijske uprave) and municipalities record the highest rates of fatal outcomes?

## Technical Methodology and Data Acquisition
Instead of manually downloading fragmented annual files, this project utilizes programmatic data acquisition via the official API of the National Open Data Portal of the Republic of Serbia (`data.gov.rs`).

### Data Source & Authenticity
* **Data Provider:** Ministry of Internal Affairs of the Republic of Serbia (Ministarstvo unutrašnjih poslova - MUP).
* **Publisher/Host:** Office for Information Technologies and Electronic Government (Kancelarija za IT i eUpravu).
* **Dataset Identifier (ID):** `5ccbf4597de2722e5424fe97` (Traffic Safety Database / База података о безбедности саобраћаја).

### Technical Approach
1. **API Integration:** We use Python's `requests` library to query the portal's backend API directly using the unique dataset ID. This bypasses the front-end interface and ensures we fetch the most up-to-date metadata.
2. **Dynamic Resource Mapping:** The script programmatically scans the dataset's metadata to discover all underlying resources (CSV/XLSX files) uploaded by MUP spanning from 2015 to 2026.
3. **Positional Stream Parsing:** Due to administrative design anomalies where individual spreadsheets lack standard text headers, files are ingested using strict positional indexing (`header=None`) to prevent horizontal dataframe fragmentation and align data columns accurately across all years.
4. **Automated Merging (In-Memory Compilation):** Python loops through the discovered download URLs, reads the data streams directly into Pandas DataFrames, and concatenates them into a single, unified historical master-dataset. This eliminates local storage clutter and automates the entire ingestion pipeline.

In [1]:
import pandas as pd
import requests
import time

# Target Dataset ID from the National Open Data Portal
dataset_id = "5ccbf4597de2722e5424fe97"
api_url = f"https://data.gov.rs/api/1/datasets/{dataset_id}/"

all_dataframes = []

try:
    print("Connecting to Open Data API and compiling annual resources...")
    response = requests.get(api_url).json()
    resources = response.get('resources', [])
    
    # Iterate through resources chronologically (from oldest 2015 to newest 2026)
    for res in reversed(resources):
        res_title = res.get('title', 'Unknown Year')
        file_url = res.get('url')
        file_format = res.get('format', '').lower()
        
        if file_format in ['xlsx', 'xls']:
            print(f"Ingesting stream: {res_title}...")
            # Enforce header=None to align data rows by strict positional index
            df_year = pd.read_excel(file_url, header=None)
            all_dataframes.append(df_year)
            time.sleep(0.4)
            
    if all_dataframes:
        # Concatenate all rows vertically
        df_traffic_all = pd.concat(all_dataframes, ignore_index=True, sort=False)
        print("\n" + "="*60)
        print("MASTER DATAFRAME COMPILED SUCCESSFULLY!")
        print(f"Total historical entries loaded: {len(df_traffic_all)}")
        print("="*60)
    else:
        print("Pipeline aborted: No valid Excel resources found.")

except Exception as e:
    print(f"Pipeline execution failure: {e}")


Connecting to Open Data API and compiling annual resources...
Ingesting stream: Подаци о саобраћајним незгодама за 2015. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2016. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2017. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2018. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2019. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2020. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2021. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
Ingesting stream: Подаци о саобраћајним незгодама за 2022. годину за територију свих ПОЛИЦИЈСКИХ УПРАВА И ОПШТИНА...
In

### Step 2: Schema Localization and Language Standardization

To prepare the multi-year consolidated database for international analytical standards, data visualization pipelines, and academic review, the positional integer indices are programmatically mapped directly to descriptive English variables. This eliminates native encoding mismatches and standardizes the features for automated analytical queries.

In [2]:
# Programmatic dictionary to map positional integers directly to English variables
column_mapping_en = {
    0: 'accident_id',
    1: 'police_directorate',
    2: 'municipality',
    3: 'timestamp',
    4: 'longitude',
    5: 'latitude',
    6: 'injury_severity',
    7: 'accident_type',
    8: 'accident_description'
}

# Execute in-memory renaming
df_traffic_all = df_traffic_all.rename(columns=column_mapping_en)

print("Database columns successfully localized to English!")
print("-" * 60)
print(df_traffic_all.info())

Database columns successfully localized to English!
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 359796 entries, 0 to 359795
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   accident_id           359796 non-null  int64  
 1   police_directorate    359796 non-null  str    
 2   municipality          359796 non-null  str    
 3   timestamp             359796 non-null  str    
 4   longitude             359796 non-null  float64
 5   latitude              359796 non-null  float64
 6   injury_severity       359796 non-null  str    
 7   accident_type         359796 non-null  str    
 8   accident_description  359796 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 24.7 MB
None


### Step 3: Temporal Feature Engineering

During the initial positional layout preview (Step 1), an audit of index `3` revealed that the Ministry of Internal Affairs (MUP) logs time metadata as a combined text string using a strict localized European format: `DD.MM.YYYY,HH:MM` (e.g., `01.10.2015,12:05`). 

To address our core research questions regarding chronological patterns—such as seasonal spikes, monthly shifts, and hourly "black windows"—this textual field must be explicitly parsed into a native Pandas datetime object. 

Once converted, we will programmatically isolate distinct features:
* `year`: To track long-term historical trends.
* `month`: To identify cyclical seasonal patterns.
* `hour`: To isolate high-risk times of day.
* `day_of_week`: To evaluate weekend vs. weekday risk distributions.

In [3]:
# Convert the string timestamp column to a native datetime object
# Format matches 'day.month.year,hour:minute' (e.g., 01.10.2015,12:05)
df_traffic_all['datetime_parsed'] = pd.to_datetime(df_traffic_all['timestamp'], format='%d.%m.%Y,%H:%M', errors='coerce')

# Extract key temporal attributes for analytical tracking
df_traffic_all['year'] = df_traffic_all['datetime_parsed'].dt.year
df_traffic_all['month'] = df_traffic_all['datetime_parsed'].dt.month
df_traffic_all['hour'] = df_traffic_all['datetime_parsed'].dt.hour
df_traffic_all['day_of_week'] = df_traffic_all['datetime_parsed'].dt.day_name()

print("Temporal feature engineering completed successfully!")
print("-" * 60)

# Check if any timestamps failed to parse (should be 0)
failed_parses = df_traffic_all['datetime_parsed'].isnull().sum()
print(f"Total unparsed/corrupted timestamps: {failed_parses}")

# Preview the newly generated temporal features
print("\nPreview of engineered time variables:")
print(df_traffic_all[['timestamp', 'year', 'month', 'hour', 'day_of_week']].head())

Temporal feature engineering completed successfully!
------------------------------------------------------------
Total unparsed/corrupted timestamps: 0

Preview of engineered time variables:
          timestamp  year  month  hour day_of_week
0  01.10.2015,12:05  2015     10    12    Thursday
1  03.10.2015,12:00  2015     10    12    Saturday
2  08.10.2015,20:00  2015     10    20    Thursday
3  11.10.2015,20:40  2015     10    20      Sunday
4  14.10.2015,19:10  2015     10    19   Wednesday


### Step 4: Initial Exploratory Data Analysis (EDA) - Temporal Risk Aggregations

With the temporal variables successfully engineered, we can now address our third research question: **How do traffic accidents fluctuate across different months and hours of the day?** 

By executing frequency aggregations (`value_counts`), we will isolate:
1. **Monthly Volume:** Highlighting seasonal spikes (e.g., summer transit months or hazardous winter conditions).
2. **Hourly "Black Windows":** Identifying the exact hours of the day when the highest volume of crashes occurs, pinpointing periods of peak risk.

In [4]:
# 1. Aggregate accidents by Month
print("Distribution of Accidents by Month (All Years Combined):")
print("-" * 60)
monthly_counts = df_traffic_all['month'].value_counts().sort_index()
for mon, count in monthly_counts.items():
    percentage = (count / len(df_traffic_all)) * 100
    print(f"Month {mon:02d} | Total Accidents: {count:,} | Share: {percentage:.2f}%")

print("\n" + "="*60 + "\n")

# 2. Aggregate accidents by Hour of the Day
print("Top 5 Most Dangerous Hours of the Day (Accident Volume):")
print("-" * 60)
hourly_ranking = df_traffic_all['hour'].value_counts()
for idx, (hr, count) in enumerate(hourly_ranking.head(5).items()):
    percentage = (count / len(df_traffic_all)) * 100
    print(f"Rank {idx+1} | Hour: {hr:02d}:00 | Total Accidents: {count:,} | Share: {percentage:.2f}%")

Distribution of Accidents by Month (All Years Combined):
------------------------------------------------------------
Month 01 | Total Accidents: 28,708 | Share: 7.98%
Month 02 | Total Accidents: 25,676 | Share: 7.14%
Month 03 | Total Accidents: 29,307 | Share: 8.15%
Month 04 | Total Accidents: 29,464 | Share: 8.19%
Month 05 | Total Accidents: 28,661 | Share: 7.97%
Month 06 | Total Accidents: 29,499 | Share: 8.20%
Month 07 | Total Accidents: 29,879 | Share: 8.30%
Month 08 | Total Accidents: 29,493 | Share: 8.20%
Month 09 | Total Accidents: 30,976 | Share: 8.61%
Month 10 | Total Accidents: 33,421 | Share: 9.29%
Month 11 | Total Accidents: 30,491 | Share: 8.47%
Month 12 | Total Accidents: 34,221 | Share: 9.51%


Top 5 Most Dangerous Hours of the Day (Accident Volume):
------------------------------------------------------------
Rank 1 | Hour: 13:00 | Total Accidents: 24,411 | Share: 6.78%
Rank 2 | Hour: 14:00 | Total Accidents: 23,987 | Share: 6.67%
Rank 3 | Hour: 15:00 | Total Accidents

### Step 5: Isolating Fatal Accidents vs. General Crash Volume

While the overall volume of traffic accidents is remarkably uniform across all months (hovering between 7% and 9.5% per month), data journalism requires us to split *volume* from *severity*. Minor urban fender-benders during afternoon rush hours (13:00–17:00) dominate the general statistics. 

To uncover the real safety crisis, we must filter our master dataset to isolate only **fatal accidents**. We will first inspect the unique values inside `injury_severity` to identify how fatal outcomes are labeled, and then aggregate those specific rows by month and hour to see if the "uniformity" pattern breaks.

In [5]:
# First, let's look at how MUP labels the severity of accidents
print("Unique values in injury_severity:")
print("-" * 60)
print(df_traffic_all['injury_severity'].value_counts())

print("\n" + "="*60 + "\n")

# Let's see if we can find rows that indicate a fatal outcome.
# We will look for terms like 'poginula', 'sa poginulim', or similar native labels.
# Once we see the exact labels in the printout above, we can run a deep slice.
print("Previewing descriptive categories to isolate fatalities:")
print(df_traffic_all['injury_severity'].unique()[:10])

Unique values in injury_severity:
------------------------------------------------------------
injury_severity
Sa mat.stetom     214405
Sa povredjenim    140133
Sa poginulim        5258
Name: count, dtype: int64


Previewing descriptive categories to isolate fatalities:
<StringArray>
['Sa mat.stetom', 'Sa povredjenim', 'Sa poginulim']
Length: 3, dtype: str


### Step 7: Longitudinal Analysis — Top 3 Deadliest Months per Year

To identify whether temporal mortality spikes are shifting over time or remaining cyclical, we execute a multi-level aggregation. We group the `df_fatal` slice by `year` and `month`, calculate total crash volumes, and isolate the top 3 high-mortality months for each individual year across the 11-year timeline.

In [6]:
# 1. Filter the master dataset to include only completed years (2015-2025)
df_fatal_complete = df_traffic_all[
    (df_traffic_all['injury_severity'] == 'Sa poginulim') & 
    (df_traffic_all['year'] <= 2025)
].copy()

# 2. Group by year and month, and count the number of fatal accidents
monthly_fatal_trends = df_fatal_complete.groupby(['year', 'month']).size().reset_index(name='fatal_accident_count')

# 3. Sort within each year to bring the highest numbers to the top
monthly_fatal_trends = monthly_fatal_trends.sort_values(by=['year', 'fatal_accident_count'], ascending=[True, False])

# 4. Extract the top 3 months for each year
top_3_months_per_year = monthly_fatal_trends.groupby('year').head(3)

print(f"Successfully verified complete timeline (2015-2025) with {len(df_fatal_complete)} records.")
print("=" * 65)
print("Top 3 Deadliest Months for Each Completed Year (2015–2025):")
print("=" * 65)

current_year = None
for idx, row in top_3_months_per_year.iterrows():
    if row['year'] != current_year:
        current_year = row['year']
        print(f"\nYear: {int(current_year)}")
        print("-" * 40)
    
    print(f"  Month: {int(row['month']):02d} | Fatal Accidents: {int(row['fatal_accident_count'])}")

Successfully verified complete timeline (2015-2025) with 5145 records.
Top 3 Deadliest Months for Each Completed Year (2015–2025):

Year: 2015
----------------------------------------
  Month: 12 | Fatal Accidents: 42
  Month: 11 | Fatal Accidents: 32
  Month: 10 | Fatal Accidents: 27

Year: 2016
----------------------------------------
  Month: 07 | Fatal Accidents: 63
  Month: 08 | Fatal Accidents: 56
  Month: 11 | Fatal Accidents: 55

Year: 2017
----------------------------------------
  Month: 10 | Fatal Accidents: 60
  Month: 09 | Fatal Accidents: 57
  Month: 08 | Fatal Accidents: 55

Year: 2018
----------------------------------------
  Month: 08 | Fatal Accidents: 64
  Month: 09 | Fatal Accidents: 58
  Month: 10 | Fatal Accidents: 45

Year: 2019
----------------------------------------
  Month: 07 | Fatal Accidents: 62
  Month: 06 | Fatal Accidents: 56
  Month: 08 | Fatal Accidents: 55

Year: 2020
----------------------------------------
  Month: 07 | Fatal Accidents: 56
  Month

### Step 8: Data Integrity Audit — Investigating the 2015 Temporal Shock

Our longitudinal analysis of high-mortality months revealed a stark anomaly: while every year from 2016 onward shows a heavy concentration of fatal accidents during the peak summer and early autumn months (June–September), the year 2015 uniquely records its spikes exclusively in October, November, and December. 

Before drawing any journalistic conclusions about 2015, we must conduct a **data integrity audit**. We will slice the master dataset for 2015 and run a full monthly distribution check to determine if the early months of that year are completely missing from the National Open Data Portal's historical repository, or if 2015 was genuinely an epidemiological outlier.

In [7]:
# Aggregate all traffic records for the year 2015 to verify monthly coverage
df_2015 = df_traffic_all[df_traffic_all['year'] == 2015]

print("Data Integrity Audit: Total recorded accidents for 2015 by month:")
print("-" * 65)

monthly_distribution_2015 = df_2015['month'].value_counts().sort_index()

if monthly_distribution_2015.empty:
    print("No data found for the year 2015.")
else:
    for month, count in monthly_distribution_2015.items():
        print(f"Month: {month:02d} | Total Recorded Accidents: {count:,}")

Data Integrity Audit: Total recorded accidents for 2015 by month:
-----------------------------------------------------------------
Month: 01 | Total Recorded Accidents: 110
Month: 02 | Total Recorded Accidents: 61
Month: 03 | Total Recorded Accidents: 77
Month: 04 | Total Recorded Accidents: 81
Month: 05 | Total Recorded Accidents: 97
Month: 06 | Total Recorded Accidents: 90
Month: 07 | Total Recorded Accidents: 123
Month: 08 | Total Recorded Accidents: 163
Month: 09 | Total Recorded Accidents: 678
Month: 10 | Total Recorded Accidents: 1,079
Month: 11 | Total Recorded Accidents: 1,240
Month: 12 | Total Recorded Accidents: 2,964


> ### ⚠️ Methodological Adjustment — Truncating the Timeline to 2016–2025
> 
> The data integrity audit confirms a critical systematic anomaly in the 2015 cohort. Total recorded incidents hover unrealistically low (fewer than 200 cases per month) between January and August, followed by an artificial exponential surge toward December. This pattern indicates that the Ministry of Internal Affairs (MUP) either implemented the digital logging pipeline mid-year or uploaded a heavily fragmented historical back-log for 2015.
> 
> To safeguard the research from structural bias — such as generating false baseline drops in historical longitudinal charts — we are making a strategic methodological decision to **exclude the year 2015 entirely**. 
> 
> Moving forward, all analytical operations, geographic profiling, and trend lines will strictly utilize the standardized, full-calendar decade spanning **2016 through 2025**.

### Step 10: Cleaned Decade Analysis (2016–2025) — High-Mortality Temporal Patterns

With the database restricted to the audited 2016–2025 timeline, we rerun our temporal analytics to extract unbiased insights regarding our third research question. 

We will isolate two critical profiles within the fatal accidents slice (`injury_severity == 'Sa poginulim'`):
1. **Top 3 Deadliest Months per Year:** To confirm if the late-summer and early-autumn spikes persist as a structural pattern across the entire decade.
2. **Top 5 Most Dangerous Hours:** To identify the exact hourly windows where fatal outcomes peak historically.

In [10]:
# 1. Isolate fatal accidents within the audited 2016-2025 dataset
df_fatal_decade = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()

# 2. Extract Top 3 Deadliest Months per Year
monthly_fatal = df_fatal_decade.groupby(['year', 'month']).size().reset_index(name='fatal_accident_count')
monthly_fatal = monthly_fatal.sort_values(by=['year', 'fatal_accident_count'], ascending=[True, False])
top_3_months = monthly_fatal.groupby('year').head(3)

print("TOP 3 DEADLIEST MONTHS PER YEAR (2016–2025)")
print("=" * 65)

current_year = None
for idx, row in top_3_months.iterrows():
    if row['year'] != current_year:
        current_year = row['year']
        print(f"\nYear: {int(current_year)}")
        print("-" * 40)
    print(f"  Month: {int(row['month']):02d} | Fatal Accidents: {int(row['fatal_accident_count'])}")

print("\n" + "="*65 + "\n")

# 3. Extract Top 5 Most Dangerous Hours for Fatal Accidents (Entire Decade)
print("TOP 5 MOST DANGEROUS HOURS FOR FATAL ACCIDENTS (2016–2025)")
print("=" * 65)

hourly_fatal_ranking = df_fatal_decade['hour'].value_counts()
total_fatalities = len(df_fatal_decade)

for idx, (hr, count) in enumerate(hourly_fatal_ranking.head(5).items()):
    percentage = (count / total_fatalities) * 100
    print(f"Rank {idx+1} | Hour: {hr:02d}:00 | Fatal Crashes: {count:,} | Share: {percentage:.2f}%")

TOP 3 DEADLIEST MONTHS PER YEAR (2016–2025)

Year: 2015
----------------------------------------
  Month: 12 | Fatal Accidents: 42
  Month: 11 | Fatal Accidents: 32
  Month: 10 | Fatal Accidents: 27

Year: 2016
----------------------------------------
  Month: 07 | Fatal Accidents: 63
  Month: 08 | Fatal Accidents: 56
  Month: 11 | Fatal Accidents: 55

Year: 2017
----------------------------------------
  Month: 10 | Fatal Accidents: 60
  Month: 09 | Fatal Accidents: 57
  Month: 08 | Fatal Accidents: 55

Year: 2018
----------------------------------------
  Month: 08 | Fatal Accidents: 64
  Month: 09 | Fatal Accidents: 58
  Month: 10 | Fatal Accidents: 45

Year: 2019
----------------------------------------
  Month: 07 | Fatal Accidents: 62
  Month: 06 | Fatal Accidents: 56
  Month: 08 | Fatal Accidents: 55

Year: 2020
----------------------------------------
  Month: 07 | Fatal Accidents: 56
  Month: 09 | Fatal Accidents: 53
  Month: 10 | Fatal Accidents: 51

Year: 2021
--------------

### Step 11: Seasonal Mortality Analysis

To further synthesize our findings, we will aggregate fatal accidents into four climatic seasons:
* **Winter:** December, January, February
* **Spring:** March, April, May
* **Summer:** June, July, August
* **Autumn:** September, October, November

This aggregation will help us quantify the **Seasonal Risk Factor**, specifically identifying if "Summer/Autumn" mortality is statistically dominant and what percentage of total annual fatalities each season accounts for.

In [11]:
# Function to map months to seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    if month in [3, 4, 5]: return 'Spring'
    if month in [6, 7, 8]: return 'Summer'
    return 'Autumn'

# Apply the mapping to our fatal accidents dataset
df_fatal_decade['season'] = df_fatal_decade['month'].apply(get_season)

# Group by year and season
seasonal_fatal = df_fatal_decade.groupby(['year', 'season']).size().reset_index(name='count')

# Calculate total fatal accidents per year to get percentage
annual_totals = df_fatal_decade.groupby('year').size().reset_index(name='annual_total')
seasonal_analysis = seasonal_fatal.merge(annual_totals, on='year')
seasonal_analysis['percentage'] = (seasonal_analysis['count'] / seasonal_analysis['annual_total']) * 100

print("SEASONAL FATALITY DISTRIBUTION (2016-2025)")
print("=" * 65)

for year in range(2016, 2026):
    print(f"\nYear: {year}")
    print("-" * 40)
    year_data = seasonal_analysis[seasonal_analysis['year'] == year]
    # Sort by count descending to see the dominant season
    for _, row in year_data.sort_values('count', ascending=False).iterrows():
        print(f"  {row['season']:<8} | Fatalities: {int(row['count']):3} | Share: {row['percentage']:.1f}%")

SEASONAL FATALITY DISTRIBUTION (2016-2025)

Year: 2016
----------------------------------------
  Summer   | Fatalities: 163 | Share: 29.3%
  Autumn   | Fatalities: 152 | Share: 27.3%
  Spring   | Fatalities: 124 | Share: 22.3%
  Winter   | Fatalities: 118 | Share: 21.2%

Year: 2017
----------------------------------------
  Autumn   | Fatalities: 157 | Share: 29.5%
  Summer   | Fatalities: 151 | Share: 28.3%
  Spring   | Fatalities: 119 | Share: 22.3%
  Winter   | Fatalities: 106 | Share: 19.9%

Year: 2018
----------------------------------------
  Autumn   | Fatalities: 145 | Share: 29.5%
  Summer   | Fatalities: 142 | Share: 28.9%
  Winter   | Fatalities: 111 | Share: 22.6%
  Spring   | Fatalities:  93 | Share: 18.9%

Year: 2019
----------------------------------------
  Summer   | Fatalities: 173 | Share: 34.0%
  Autumn   | Fatalities: 148 | Share: 29.1%
  Spring   | Fatalities: 103 | Share: 20.2%
  Winter   | Fatalities:  85 | Share: 16.7%

Year: 2020
-----------------------------

### Step 12: Investigative Audit — Determining Accident Causes

To transition from "when" to "how," we need to understand the mechanism behind the fatal incidents. The dataset provides two descriptive columns: `accident_type` and `accident_description`. 

Before proceeding with a deeper cross-analysis of the most dangerous months, I am conducting a brief audit of these two columns. By inspecting the unique values and a sample of entries, I will determine which column provides the most granular and reliable data regarding the actual cause or nature of the fatal event. This will ensure that our future "Top Cause" rankings are based on the most accurate source of information.

In [12]:
# Inspecting the content of the two potential columns to evaluate their analytical value
print("AUDIT: Examining 'accident_type' vs 'accident_description'")
print("=" * 70)

# Show unique values for accident_type
print("\nUnique values in 'accident_type':")
print(df_fatal_decade['accident_type'].unique())

# Show a few sample entries from accident_description to understand the depth of detail
print("\nSample entries from 'accident_description':")
print(df_fatal_decade['accident_description'].head(10).tolist())

# Check for null values to ensure data quality
print("\nMissing values check:")
print(f"accident_type: {df_fatal_decade['accident_type'].isnull().sum()}")
print(f"accident_description: {df_fatal_decade['accident_description'].isnull().sum()}")

AUDIT: Examining 'accident_type' vs 'accident_description'

Unique values in 'accident_type':
<StringArray>
[                                    'SN SA PEŠACIMA',
          'SN SA NAJMANjE DVA VOZILA – BEZ SKRETANjA',
                               'SN SA JEDNIM VOZILOM',
 'SN SA NAJMANjE DVA VOZILA – SKRETANjE ILI PRELAZAK',
                          'SN SA PARKIRANIM VOZILIMA']
Length: 5, dtype: str

Sample entries from 'accident_description':
['Pešak se kreće duž kolovoza', 'Ostale nezgode sa učešćem najmanje dva vozila bez skretanja (bez podataka o smeru)', 'Ostale nezgode sa učešćem najmanje dva vozila bez skretanja', 'Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini', 'Prelazak pešaka zdesna, van raskrsnice, bez skretanja vozila', 'Nezgoda sa jednim vozilom – silazak ulevo sa kolovoza na pravcu', 'Prelazak pešaka sleva, sa skretanjem vozila ulevo, u raskrsnici', 'Najmanje dva vozila – suprotni smerovi bez skretanja – kretanje unazad', 'Prelazak pešaka zdesna, sa skretanj

### Step 13: Analyzing Primary Mechanisms of Fatalities

Having audited the dataset, we have identified `accident_description` as the primary repository for understanding the mechanism of fatal accidents. 

In this step, we map the recurring patterns within our "Top 3 Deadliest Months" per year. By aggregating the text-based descriptions, we aim to extract the three most frequent types of incidents that lead to fatal outcomes. This will reveal whether specific months are dominated by recurring behaviors (e.g., speed-related incidents, loss of vehicle control, or specific types of collisions).

In [13]:
# Analysis of Top 3 Causes (accident_description) in the Top 3 Deadliest Months per Year

print("TOP 3 FATAL ACCIDENT MECHANISMS IN HIGH-RISK MONTHS (2016–2025)")
print("=" * 75)

for year in range(2016, 2026):
    year_data = df_fatal_decade[df_fatal_decade['year'] == year]
    
    # Identify the top 3 months for this specific year
    top_months = year_data.groupby('month').size().nlargest(3).index
    
    print(f"\nYear: {year}")
    print("-" * 50)
    
    for month in top_months:
        month_data = year_data[year_data['month'] == month]
        
        # Get the top 3 most frequent descriptions
        top_descriptions = month_data['accident_description'].value_counts().head(3)
        
        print(f"  Month {int(month):02d}:")
        for desc, count in top_descriptions.items():
            print(f"    - {desc} ({count} cases)")

TOP 3 FATAL ACCIDENT MECHANISMS IN HIGH-RISK MONTHS (2016–2025)

Year: 2016
--------------------------------------------------
  Month 07:
    - Najmanje dva vozila – čeoni sudar (10 cases)
    - Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu (7 cases)
    - Najmanje dva vozila koja se kreću u istom smeru – sustizanje (7 cases)
  Month 08:
    - Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini (8 cases)
    - Nezgoda sa jednim vozilom – silazak ulevo sa kolovoza na pravcu (6 cases)
    - Ostale nezgode sa najmanje dva vozila – suprotni smerovi bez skretanja (6 cases)
  Month 11:
    - Prelazak pešaka sleva, van raskrsnice , bez skretanja vozila (9 cases)
    - Prelazak pešaka zdesna, van raskrsnice, bez skretanja vozila (5 cases)
    - Najmanje dva vozila – čeoni sudar (5 cases)

Year: 2017
--------------------------------------------------
  Month 10:
    - Ostale nezgode sa najmanje dva vozila – suprotni smerovi bez skretanja (6 cases)
    - Najmanje dva vo

### Step 14: Macro-Analysis of Primary Accident Mechanisms (2016–2025)

After identifying the most dangerous months and their specific patterns, we now widen the lens to the entire decade. In this step, we calculate the frequency and percentage share of the top 5 most common accident mechanisms recorded in the `accident_description` column. 

This macro-analysis will provide a definitive ranking of the structural risks present on Serbian roads, allowing us to distinguish between predominant crash types and less frequent anomalies.

In [ ]:
# Calculate the total count and percentage of the top 5 accident mechanisms
top_5_descriptions = df_analysis['accident_description'].value_counts().head(5)
total_records = len(df_analysis)

print("TOP 5 MOST FREQUENT ACCIDENT MECHANISMS (2016–2025)")
print("=" * 70)
print(f"{'Mechanism':<50} | {'Count':<10} | {'Share (%)':<10}")
print("-" * 75)

for desc, count in top_5_descriptions.items():
    percentage = (count / total_records) * 100
    print(f"{desc:<50} | {count:<10,} | {percentage:>8.2f}%")

### Step 15: Macro-Analysis of Fatal Accident Mechanisms (2016–2025)

To sharpen our investigative focus, we now filter the dataset exclusively for fatal incidents (`injury_severity == 'Sa poginulim'`). 

This macro-analysis identifies the five most frequent accident mechanisms associated with fatalities over the last decade. By comparing these results with the general accident data, we can isolate which types of traffic events are disproportionately lethal, highlighting the primary "kill factors" on Serbian roads.

In [15]:
# Filter specifically for fatal accidents
df_fatal_only = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()

# Calculate the top 5 mechanisms for fatalities
top_5_fatal_mechanisms = df_fatal_only['accident_description'].value_counts().head(5)
total_fatal_records = len(df_fatal_only)

print("TOP 5 MOST FREQUENT MECHANISMS IN FATAL ACCIDENTS (2016–2025)")
print("=" * 80)
print(f"{'Mechanism':<55} | {'Count':<10} | {'Share (%)':<10}")
print("-" * 80)

for desc, count in top_5_fatal_mechanisms.items():
    percentage = (count / total_fatal_records) * 100
    print(f"{desc:<55} | {count:<10,} | {percentage:>8.2f}%")

TOP 5 MOST FREQUENT MECHANISMS IN FATAL ACCIDENTS (2016–2025)
Mechanism                                               | Count      | Share (%) 
--------------------------------------------------------------------------------
Najmanje dva vozila – čeoni sudar                       | 512        |     9.74%
Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu | 484        |     9.21%
Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini | 381        |     7.25%
Najmanje dva vozila koja se kreću u istom smeru – sustizanje | 377        |     7.17%
Nezgoda sa jednim vozilom – silazak ulevo sa kolovoza na pravcu | 357        |     6.79%


In [17]:
# Helper function for percentage calculation
def get_perc(part, total):
    return (part / total) * 100 if total > 0 else 0

total_all = len(df_traffic_all)
total_fatal = len(df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'])

print("--- OVERALL STATISTICS (2016-2025) ---")
print(f"Total accidents: {total_all:,}")
print(f"Fatal accidents: {total_fatal:,} ({get_perc(total_fatal, total_all):.2f}%)")

# 1. By Year
print("\n--- BY YEAR ---")
for y in range(2016, 2026):
    y_data = df_traffic_all[df_traffic_all['year'] == y]
    y_all = len(y_data)
    y_fat = len(y_data[y_data['injury_severity'] == 'Sa poginulim'])
    print(f"{y}: Total {y_all}, Fatal {y_fat} ({get_perc(y_fat, y_all):.2f}%)")

# 2. Top 5 Causes
print("\n--- TOP 5 CAUSES (ALL VS FATAL) ---")
for lbl, df_sub in [("ALL", df_traffic_all), ("FATAL", df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'])]:
    print(f"\n{lbl}:")
    top5 = df_sub['accident_description'].value_counts().head(5)
    for desc, count in top5.items():
        print(f"  {desc}: {count} ({get_perc(count, len(df_sub)):.2f}%)")

# 3. Top 3 Months
print("\n--- TOP 3 MONTHS ---")
for lbl, df_sub in [("ALL", df_traffic_all), ("FATAL", df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'])]:
    print(f"\n{lbl}:")
    top3_m = df_sub['month'].value_counts().head(3)
    for m, count in top3_m.items():
        print(f"  Month {m}: {count} ({get_perc(count, len(df_sub)):.2f}%)")

# 4. Top 5 Hours
print("\n--- TOP 5 HOURS ---")
for lbl, df_sub in [("ALL", df_traffic_all), ("FATAL", df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'])]:
    print(f"\n{lbl}:")
    top5_h = df_sub['hour'].value_counts().head(5)
    for h, count in top5_h.items():
        print(f"  {h:02d}:00 - {count} ({get_perc(count, len(df_sub)):.2f}%)")

# 5. Top 5 Municipalities
print("\n--- TOP 5 MUNICIPALITIES BY YEAR ---")
for y in range(2016, 2026):
    y_data = df_traffic_all[df_traffic_all['year'] == y]
    print(f"\nYear {y}:")
    print("  Top locations (All):")
    for loc, count in y_data['municipality'].value_counts().head(5).items():
        print(f"    {loc}: {count}")
    print("  Top locations (Fatal):")
    fatal_y = y_data[y_data['injury_severity'] == 'Sa poginulim']
    for loc, count in fatal_y['municipality'].value_counts().head(5).items():
        print(f"    {loc}: {count}")

--- OVERALL STATISTICS (2016-2025) ---
Total accidents: 359,796
Fatal accidents: 5,258 (1.46%)

--- BY YEAR ---
2016: Total 36165, Fatal 557 (1.54%)
2017: Total 36672, Fatal 533 (1.45%)
2018: Total 35993, Fatal 491 (1.36%)
2019: Total 35956, Fatal 509 (1.42%)
2020: Total 30891, Fatal 470 (1.52%)
2021: Total 34751, Fatal 490 (1.41%)
2022: Total 33399, Fatal 517 (1.55%)
2023: Total 32974, Fatal 475 (1.44%)
2024: Total 32330, Fatal 485 (1.50%)
2025: Total 33415, Fatal 463 (1.39%)

--- TOP 5 CAUSES (ALL VS FATAL) ---

ALL:
  Najmanje dva vozila koja se kreću u istom smeru – sustizanje: 49652 (13.80%)
  Sudar sa parkiranim vozilom sa desne strane kolovoza: 23713 (6.59%)
  Najmanje dva vozila koja se kreću različitim putevima uz prolazak kroz raskrsnicu, ili od kojih jedno prelazi preko kolovoza, bez skretanja: 18866 (5.24%)
  Ostale nezgode sa najmanje dva vozila – suprotni smerovi bez skretanja: 17345 (4.82%)
  Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu: 16124 (4.48%)

In [21]:
# Filter for fatal accidents only first
fatal_df = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()

# Filter for single-vehicle accidents within fatal ones
# We use case=False to be safe with capitalization
single_vehicle_fatal = fatal_df[fatal_df['accident_description'].str.contains('Nezgoda sa jednim vozilom', case=False, na=False)]

# Calculations
count_single = len(single_vehicle_fatal)
count_total_fatal = len(fatal_df)
percentage_single = (count_single / count_total_fatal) * 100

print("--- SINGLE-VEHICLE FATAL ACCIDENTS ANALYSIS ---")
print(f"Total fatal accidents: {count_total_fatal:,}")
print(f"Single-vehicle fatal accidents: {count_single:,}")
print(f"Share of single-vehicle accidents in total fatal outcomes: {percentage_single:.2f}%")

# Bonus: Let's see the sub-types of these single-vehicle accidents
print("\n--- BREAKDOWN OF SINGLE-VEHICLE FATAL ACCIDENTS ---")
print(single_vehicle_fatal['accident_description'].value_counts())

--- SINGLE-VEHICLE FATAL ACCIDENTS ANALYSIS ---
Total fatal accidents: 5,258
Single-vehicle fatal accidents: 1,600
Share of single-vehicle accidents in total fatal outcomes: 30.43%

--- BREAKDOWN OF SINGLE-VEHICLE FATAL ACCIDENTS ---
accident_description
Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu                        484
Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini                               381
Nezgoda sa jednim vozilom – silazak ulevo sa kolovoza na pravcu                         357
Nezgoda sa jednim vozilom i prevrtanjem                                                 239
Nezgoda sa jednim vozilom na kolovozu                                                   108
Nezgoda sa jednim vozilom bez prepreka na kolovozu na nepoznat, nespecificiran način     31
Name: count, dtype: int64


In [22]:
# Extract all unique accident descriptions and their counts
# This will show us how fragmented the descriptions are
all_causes = df_traffic_all['accident_description'].value_counts()

print("ALL UNIQUE ACCIDENT DESCRIPTIONS AND THEIR FREQUENCY")
print("=" * 70)
# Showing first 50 to get an idea of the variety
for desc, count in all_causes.head(50).items():
    print(f"{desc}: {count}")

print(f"\nTotal unique descriptions found: {len(all_causes)}")

ALL UNIQUE ACCIDENT DESCRIPTIONS AND THEIR FREQUENCY
Najmanje dva vozila koja se kreću u istom smeru – sustizanje: 49652
Sudar sa parkiranim vozilom sa desne strane kolovoza: 23713
Najmanje dva vozila koja se kreću različitim putevima uz prolazak kroz raskrsnicu, ili od kojih jedno prelazi preko kolovoza, bez skretanja: 18866
Ostale nezgode sa najmanje dva vozila – suprotni smerovi bez skretanja: 17345
Nezgoda sa jednim vozilom – silazak udesno sa kolovoza na pravcu: 16124
Najmanje dva vozila koja se kreću istim putem u suprotnim smerovima uz skretanje ulevo ispred drugog vozila: 15209
Nezgoda sa jednim vozilom na kolovozu: 13850
Ostali sudari sa parkiranim vozilom: 12429
Najmanje dva vozila koja se kreću istim putem u istom smeru uz skretanje, skretanje ulevo ispred drugog vozila: 11713
Nezgoda sa jednim vozilom – silazak ulevo sa kolovoza na pravcu: 11607
Nezgoda sa jednim vozilom – silazak sa kolovoza u krivini: 10260
Sudar sa parkiranim vozilom sa leve strane kolovoza: 9808
Nezgode

In [23]:
# Filter for fatal accidents
fatal_df = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()

# Filter for accidents involving at least two vehicles
# The search string 'Najmanje dva vozila' covers various collision types
multi_vehicle_fatal = fatal_df[fatal_df['accident_description'].str.contains('Najmanje dva vozila', case=False, na=False)]

# Statistics
count_multi = len(multi_vehicle_fatal)
count_total_fatal = len(fatal_df)
percentage_multi = (count_multi / count_total_fatal) * 100

print("--- FATAL ACCIDENTS INVOLVING AT LEAST TWO VEHICLES ---")
print(f"Total fatal accidents: {count_total_fatal:,}")
print(f"Fatal multi-vehicle accidents: {count_multi:,}")
print(f"Share in total fatal outcomes: {percentage_multi:.2f}%")

# Preview the sub-types to verify accuracy
print("\n--- TYPES OF MULTI-VEHICLE FATAL ACCIDENTS ---")
print(multi_vehicle_fatal['accident_description'].value_counts().head(10))

--- FATAL ACCIDENTS INVOLVING AT LEAST TWO VEHICLES ---
Total fatal accidents: 5,258
Fatal multi-vehicle accidents: 2,089
Share in total fatal outcomes: 39.73%

--- TYPES OF MULTI-VEHICLE FATAL ACCIDENTS ---
accident_description
Najmanje dva vozila – čeoni sudar                                                                                                              512
Najmanje dva vozila koja se kreću u istom smeru – sustizanje                                                                                   377
Najmanje dva vozila koja se kreću istim putem u suprotnim smerovima uz skretanje ulevo ispred drugog vozila                                    283
Ostale nezgode sa najmanje dva vozila – suprotni smerovi bez skretanja                                                                         207
Najmanje dva vozila koja se kreću različitim putevima uz prolazak kroz raskrsnicu, ili od kojih jedno prelazi preko kolovoza, bez skretanja    124
Najmanje dva vozila koja se kreću is

In [ ]:
# Filter for fatal accidents
fatal_df = df_analysis[df_analysis['injury_severity'] == 'Sa poginulim'].copy()

# Filter for accidents involving pedestrians
# We use 'pešak' to capture various forms of the word (pešak, pešaka, pešaku...)
pedestrian_fatal = fatal_df[fatal_df['accident_description'].str.contains('pešak', case=False, na=False)]

# Calculations
count_pedestrian = len(pedestrian_fatal)
count_total_fatal = len(fatal_df)
percentage_pedestrian = (count_pedestrian / count_total_fatal) * 100

print("--- FATAL ACCIDENTS INVOLVING PEDESTRIANS ---")
print(f"Total fatal accidents: {count_total_fatal:,}")
print(f"Fatal accidents involving a pedestrian: {count_pedestrian:,}")
print(f"Share in total fatal outcomes: {percentage_pedestrian:.2f}%")

# Preview of specific descriptions to see the context (e.g., crossing, highway, etc.)
print("\n--- TYPES OF FATAL PEDESTRIAN ACCIDENTS ---")
print(pedestrian_fatal['accident_description'].value_counts().head(10))

In [24]:
# Helper function for monthly analysis
def get_top_months(df, category_name):
    total = len(df)
    # Get top 12 months
    top_months = df['month'].value_counts().head(12)
    
    print(f"\n--- {category_name} (Top 12 Months) ---")
    for m, count in top_months.items():
        print(f"  Month {m}: {count} ({get_perc(count, total):.2f}%)")

# Define DataFrames
all_fatal = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim']
single_veh = all_fatal[all_fatal['accident_description'].str.contains('Nezgoda sa jednim vozilom', case=False, na=False)]
multi_veh = all_fatal[all_fatal['accident_description'].str.contains('Najmanje dva vozila', case=False, na=False)]
ped_fatal = all_fatal[all_fatal['accident_description'].str.contains('pešak', case=False, na=False)]

# Run Analysis for months only
categories = [
    ("TOTAL ACCIDENTS", df_traffic_all),
    ("FATAL ACCIDENTS", all_fatal),
    ("FATAL SINGLE-VEHICLE", single_veh),
    ("FATAL MULTI-VEHICLE", multi_veh),
    ("FATAL PEDESTRIAN", ped_fatal)
]

for name, df in categories:
    get_top_months(df, name)


--- TOTAL ACCIDENTS (Top 12 Months) ---
  Month 12: 34221 (9.51%)
  Month 10: 33421 (9.29%)
  Month 9: 30976 (8.61%)
  Month 11: 30491 (8.47%)
  Month 7: 29879 (8.30%)
  Month 6: 29499 (8.20%)
  Month 8: 29493 (8.20%)
  Month 4: 29464 (8.19%)
  Month 3: 29307 (8.15%)
  Month 1: 28708 (7.98%)
  Month 5: 28661 (7.97%)
  Month 2: 25676 (7.14%)

--- FATAL ACCIDENTS (Top 12 Months) ---
  Month 9: 530 (10.08%)
  Month 8: 519 (9.87%)
  Month 10: 510 (9.70%)
  Month 7: 502 (9.55%)
  Month 12: 474 (9.01%)
  Month 11: 466 (8.86%)
  Month 6: 435 (8.27%)
  Month 3: 381 (7.25%)
  Month 5: 381 (7.25%)
  Month 1: 375 (7.13%)
  Month 4: 366 (6.96%)
  Month 2: 319 (6.07%)

--- FATAL SINGLE-VEHICLE (Top 12 Months) ---
  Month 7: 193 (12.06%)
  Month 8: 180 (11.25%)
  Month 9: 162 (10.12%)
  Month 10: 153 (9.56%)
  Month 6: 150 (9.38%)
  Month 4: 126 (7.88%)
  Month 12: 120 (7.50%)
  Month 11: 116 (7.25%)
  Month 5: 112 (7.00%)
  Month 3: 110 (6.88%)
  Month 1: 103 (6.44%)
  Month 2: 75 (4.69%)

--- FAT

In [25]:
import pandas as pd

# 1. Prepare data
fatal_df = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()
# Ensure timestamp is datetime
fatal_df['timestamp'] = pd.to_datetime(fatal_df['timestamp'])
fatal_df['day_of_year'] = fatal_df['timestamp'].dt.dayofyear.sort_values()

# Total number of fatal accidents
total_fatal = len(fatal_df)
# Set target to 33%
target = total_fatal * 0.33

# 2. Sliding window algorithm
all_days = sorted(fatal_df['day_of_year'].tolist())
extended_days = all_days + [d + 365 for d in all_days]

min_window_size = 366
best_start = -1
best_end = -1

# Sliding window
for i in range(len(all_days)):
    for j in range(i, len(all_days)):
        count = j - i + 1
        if count >= target:
            window_size = all_days[j] - all_days[i]
            
            if window_size < min_window_size:
                min_window_size = window_size
                best_start = all_days[i]
                best_end = all_days[j]
            break

# 3. Output results
start_date = pd.Timestamp(2025, 1, 1) + pd.Timedelta(days=best_start-1)
end_date = pd.Timestamp(2025, 1, 1) + pd.Timedelta(days=best_end-1)

print(f"The shortest period covering 33% of all fatal accidents lasts {min_window_size} days.")
print(f"Period starts: {start_date.strftime('%d. %B')}")
print(f"Period ends: {end_date.strftime('%d. %B')}")

C:\Users\startit\AppData\Local\Temp\ipykernel_13748\970579796.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fatal_df['timestamp'] = pd.to_datetime(fatal_df['timestamp'])


The shortest period covering 33% of all fatal accidents lasts 106 days.
Period starts: 15. July
Period ends: 29. October


In [26]:
import pandas as pd

# Define the two datasets
all_accidents = df_traffic_all.copy()
fatal_accidents = df_traffic_all[df_traffic_all['injury_severity'] == 'Sa poginulim'].copy()

# 1. Total accidents per hour (Ranking)
total_by_hour = all_accidents['hour'].value_counts().reset_index()
total_by_hour.columns = ['Hour', 'Total_Accidents']
total_by_hour = total_by_hour.sort_values(by='Total_Accidents', ascending=False)

# 2. Fatal accidents per hour (Ranking)
fatal_by_hour = fatal_accidents['hour'].value_counts().reset_index()
fatal_by_hour.columns = ['Hour', 'Fatal_Accidents']
fatal_by_hour = fatal_by_hour.sort_values(by='Fatal_Accidents', ascending=False)

print("--- HOURS WITH HIGHEST TOTAL NUMBER OF ACCIDENTS ---")
print(total_by_hour.to_string(index=False))

print("\n--- HOURS WITH HIGHEST NUMBER OF FATAL ACCIDENTS ---")
print(fatal_by_hour.to_string(index=False))

--- HOURS WITH HIGHEST TOTAL NUMBER OF ACCIDENTS ---
 Hour  Total_Accidents
   13            24411
   14            23987
   15            23940
   16            23861
   17            23064
   12            22339
   18            21182
   11            21061
   10            19050
   19            18788
    9            16715
   20            16273
    8            15581
   21            14055
    7            13964
   22            12079
   23             9132
    6             8826
    0             7449
    1             6217
    2             5024
    5             4570
    3             4433
    4             3795

--- HOURS WITH HIGHEST NUMBER OF FATAL ACCIDENTS ---
 Hour  Fatal_Accidents
   17              376
   18              363
   20              312
   19              311
   16              289
   15              279
   14              256
   21              254
   13              251
   12              249
   11              224
   22              221
    6              

### Step 16: Make data tables

Based on the data extracted so far, I have drafted the article text and established the visual structure. I have designed the article to include three data tables and one interactive clock:

* **Table 1:** Provides general accident statistics.
* **Table 2:** Focuses on accident participants.
* **Table 3:** Analyzes monthly accident trends and types.
* **Interactive Clock:** Displays the average number of accidents per hour.

In [35]:
# [STEP 1] Define accident subsets (filters)
single_veh = df_traffic_all[df_traffic_all['accident_description'].str.contains('jednim vozilom', case=False, na=False)]
multi_veh = df_traffic_all[df_traffic_all['accident_description'].str.contains('Najmanje dva vozila', case=False, na=False)]
ped_df = df_traffic_all[df_traffic_all['accident_description'].str.contains('pešak|pešac', case=False, na=False)]

# [TABLE 1] General Accident Statistics
table1 = df_traffic_all.groupby('year').agg(
    Total_Accidents=('accident_id', 'count'),
    Fatal_Accidents=('injury_severity', lambda x: (x == 'Sa poginulim').sum())
)
table1.to_csv('table_1_general_stats.csv')

# [TABLE 2] Accident Participants Analysis - Function
def get_participant_stats_fixed(df, label):
    df_temp = df.copy()
    df_temp['Fatal'] = (df_temp['injury_severity'] == 'Sa poginulim').astype(int)
    df_temp['NonFatal'] = (df_temp['injury_severity'] != 'Sa poginulim').astype(int)
    
    # Group by year and sum
    stats = df_temp.groupby('year')[['NonFatal', 'Fatal']].sum()
    stats.columns = [f'{label}_NonFatal', f'{label}_Fatal']
    return stats

# Generate and save Table 2
table2 = pd.concat([
    get_participant_stats_fixed(single_veh, 'Single'), 
    get_participant_stats_fixed(multi_veh, 'Multi'), 
    get_participant_stats_fixed(ped_df, 'Pedestrian')
], axis=1).fillna(0)

table2.to_csv('table_2_participant_analysis.csv')



# [TABLE 3] Monthly Seasonal Analysis - Outcome-based
# We use .str.strip() to remove any invisible whitespace and .str.lower() to avoid case issues

def get_monthly_outcome_stats(is_fatal):
    # Standardize the column for comparison
    severity = df_traffic_all['injury_severity'].astype(str).str.strip()
    
    # Define the filter condition
    if is_fatal:
        # Check for various common ways 'Sa poginulim' might appear
        df_outcome = df_traffic_all[severity.str.contains('poginulim', case=False, na=False)].copy()
    else:
        # Check for everything else
        df_outcome = df_traffic_all[~severity.str.contains('poginulim', case=False, na=False)].copy()
    
    # Categorize types dynamically
    df_outcome['Type'] = 'Other'
    df_outcome.loc[df_outcome['accident_description'].str.contains('jednim vozilom', case=False, na=False), 'Type'] = 'Single_Vehicle'
    df_outcome.loc[df_outcome['accident_description'].str.contains('Najmanje dva vozila', case=False, na=False), 'Type'] = 'Multi_Vehicle'
    df_outcome.loc[df_outcome['accident_description'].str.contains('pešak|pešac', case=False, na=False), 'Type'] = 'Pedestrian'
    
    return df_outcome.groupby(['month', 'Type']).size().unstack(fill_value=0)

# Generate tables
fatal_seasonal = get_monthly_outcome_stats(True)
non_fatal_seasonal = get_monthly_outcome_stats(False)

# Save
fatal_seasonal.to_csv('table_3_seasonal_fatal.csv')
non_fatal_seasonal.to_csv('table_3_seasonal_non_fatal.csv')

print("Tables regenerated with improved filtering.")

Tables regenerated with improved filtering.


In [51]:
# 1. Load data
with open('clock_data.json', 'r') as f:
    data = json.load(f)

# 2. Convert to DataFrame
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = 'Hour'
df.reset_index(inplace=True)

# 3. Calculate percentages correctly: 
# Svaki sat je procenat u odnosu na SUMU TE KOLONE
df['Fatal_Pct'] = (df['Fatal'] / df['Fatal'].sum()) * 100
df['Non_Fatal_Pct'] = (df['Non_Fatal'] / df['Non_Fatal'].sum()) * 100

# 4. Prepare clean file
final_df = df[['Hour', 'Fatal_Pct', 'Non_Fatal_Pct', 'Fatal', 'Non_Fatal']]
final_df.to_csv('hourly_distribution_final.csv', index=False)

print("Spreman CSV sa nezavisnim procentima!")

Spreman CSV sa nezavisnim procentima!
